# Graph machine learning: prepare before predicting

**Atlas source reading copy · browser-local**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Urbanity/blob/a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f/book/notebooks/graph-ml.ipynb)



Standard-library synthetic network, context, imagery-label and graph exercises; these executable cells do not import Urbanity or download OSM/provider data. Native Urbanity, compiled GIS, imagery and graph-ML examples in Markdown are continuations requiring their own environment and data permissions. Owner-authorized original material has no inferred public repository license.

**Provenance:** `book/notebooks/graph-ml.ipynb` at `a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f`; original SHA-256 `3ac7777bde9db0673d4267c937eded1e3d02100f13116911fcf50b31218763a3`. Original owner-authored companion incorporated at repository owner request; Urbanity source repository has no public license grant. Package Urbanity and third-party data retain separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Graph machine learning: prepare before predicting

A graph-learning task combines node features, labelled examples, and relationships. Before training, define the prediction target and split strategy; watch for spatial leakage and class imbalance. This browser lesson demonstrates only a tiny, synthetic neighbour-label baseline in standard Python. It does not train a machine-learning model and does not use a city dataset.

[Launch this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Urbanity/lab/lab/index.html?path=notebooks/graph-ml.ipynb) · [Official Urbanity graph-ML tutorial](https://urbanity.readthedocs.io/en/latest/tutorials/graph-ml/)

## Try it: a transparent neighbourhood baseline

Each node has a toy class label, and the graph says which nodes are adjacent. For a held-out node, predict the most frequent label among its training neighbours. This deliberately simple baseline reveals how labels can depend on graph structure.

In [1]:
labels = {"a": "home", "b": "home", "c": "shop", "d": "shop", "e": "home"}
adjacency = {
    "a": {"b", "c"}, "b": {"a", "c", "d"}, "c": {"a", "b", "d"},
    "d": {"b", "c", "e"}, "e": {"d"},
}
train_nodes = {"a", "b", "c", "d"}
test_nodes = {"e"}

def neighbour_vote(node):
    votes = {}
    for neighbour in adjacency[node] & train_nodes:
        label = labels[neighbour]
        votes[label] = votes.get(label, 0) + 1
    if not votes:
        return None
    return sorted(votes, key=lambda label: (-votes[label], label))[0]

predictions = {node: neighbour_vote(node) for node in sorted(test_nodes)}
for node, prediction in predictions.items():
    print(node, "true:", labels[node], "predicted:", prediction)

e true: home predicted: shop


This baseline is not evidence that neighbouring buildings should have the same use. A serious evaluation compares against non-graph baselines, checks label and feature provenance, creates geographically appropriate train/validation/test splits, and reports performance by class and place.

## Native Urbanity → graph-learning workflow

Urbanity documents conversion of an `UrbanGraph` into graph-learning formats, including PyTorch Geometric. This needs the native environment and compatible optional ML dependencies. The example below prepares data; consult the upstream tutorial for the full, version-specific training workflow.

```python
urban_graph.initialize_edges(building_neighbours="knn", knn=3, distance=100)
data = urban_graph.to_pyg_graph(
    target_node="building",
    categorical=True,
    train_val_test=[0.8, 0.1, 0.1],
    random_seed=0,
    target_value=building_labels,
)
```

Review target and feature leakage, split design, class balance, spatial autocorrelation, and ethical implications before fitting a model. See [Urbanity graph ML](https://urbanity.readthedocs.io/en/latest/tutorials/graph-ml/) and the [UrbanGraph API](https://urbanity.readthedocs.io/en/latest/api/data_class/).

## Reflect

- Is the held-out node truly independent of the training data?
- What baseline would you compare against if graph links were ignored?
- Could labels or model features encode sensitive information or reproduce bias?